# Spark DataFrames & Spark SQL - Complete Solution Notebook

In [1]:
# Scenario Statement: Global Streaming Platform OptimizationThe Context: You are a Data Engineer at a global video
# streaming service (like Netflix or YouTube). Every second, your servers generate logs for millions of active video streams.
# These logs contain the "Bandwidth Allocation" (how much data is being sent) for every single user currently watching a video.

# The Problem: Due to a new "Ultra-HD" initiative, the company needs to double the bandwidth allocation for every active stream
#  immediately to improve video quality.

# The Challenge:Massive Data: The number of logs is in the millions, making it impossible for
#   a single computer to process them without crashing or taking hours.

# # Real-Time Speed: The infrastructure costs need to be calculated instantly so server capacity can be adjusted in real-time.
# Mapping the Scenario to the CodeCode StepScenario



from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LabSolution").getOrCreate()

## Load Dataset

In [ ]:
df = spark.read.csv("/content/spark_lab_dataset.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()

+--------------+-----------+----------------+------+--------+----------------+--------+
|transaction_id|customer_id|            name|region|  amount|transaction_type|is_fraud|
+--------------+-----------+----------------+------+--------+----------------+--------+
|             1|       1057|Christian Forbes| South|12935.47|        Purchase|       0|
|             2|       1003|   Justin Harris| South|  3709.3|      Withdrawal|       0|
|             3|       1004|   Peter Schmitt| South| 3837.45|        Purchase|       1|
|             4|       1049|  Molly Gonzalez|  East| 46638.1|      Withdrawal|       1|
|             5|       1045| Wanda Mcfarland| North|24212.31|        Purchase|       1|
+--------------+-----------+----------------+------+--------+----------------+--------+
only showing top 5 rows
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- amount: d

## Data Exploration

This cell performs basic data exploration to understand the dataset. It calculates the total number of transactions, lists distinct regions, and counts the occurrences of each transaction type.

In [ ]:
print("Total Transactions:", df.count())
df.select("region").distinct().show()
df.groupBy("transaction_type").count().show()

Total Transactions: 500
+------+
|region|
+------+
| South|
|  East|
|  West|
| North|
+------+

+----------------+-----+
|transaction_type|count|
+----------------+-----+
|        Purchase|  162|
|          Refund|  172|
|      Withdrawal|  166|
+----------------+-----+



This cell performs basic data exploration to understand the dataset. It calculates the total number of transactions, lists distinct regions, and counts the occurrences of each transaction type.

## Data Cleaning

This cell cleans the data by dropping any rows that contain null values using `df.dropna()`. It then prints the schema again to confirm that the DataFrame structure remains consistent after cleaning.

In [ ]:
df = df.dropna()
df.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- is_fraud: integer (nullable = true)



This cell cleans the data by dropping any rows that contain null values using `df.dropna()`. It then prints the schema again to confirm that the DataFrame structure remains consistent after cleaning.

## Transformations

This cell demonstrates various data transformations. It filters the DataFrame to create subsets for 'high_value' transactions (amount > 10000) and 'fraud_df' (where 'is_fraud' is 1). It also uses `select()` to create a DataFrame with specific columns: customer_id, amount, and region.

In [ ]:
high_value = df.filter(df.amount > 10000)
fraud_df = df.filter(df.is_fraud == 1)
selected = df.select("customer_id", "amount", "region")

high_value.show(5)
fraud_df.show(5)
selected.show(5)

+--------------+-----------+----------------+------+--------+----------------+--------+
|transaction_id|customer_id|            name|region|  amount|transaction_type|is_fraud|
+--------------+-----------+----------------+------+--------+----------------+--------+
|             1|       1057|Christian Forbes| South|12935.47|        Purchase|       0|
|             4|       1049|  Molly Gonzalez|  East| 46638.1|      Withdrawal|       1|
|             5|       1045| Wanda Mcfarland| North|24212.31|        Purchase|       1|
|             6|       1045|  Jennifer Casey| South|35929.02|      Withdrawal|       1|
|             7|       1054|   Melissa Klein| North|26482.51|        Purchase|       1|
+--------------+-----------+----------------+------+--------+----------------+--------+
only showing top 5 rows
+--------------+-----------+---------------+------+--------+----------------+--------+
|transaction_id|customer_id|           name|region|  amount|transaction_type|is_fraud|
+---------

This cell demonstrates various data transformations. It filters the DataFrame to create subsets for 'high_value' transactions (amount > 10000) and 'fraud_df' (where 'is_fraud' is 1). It also uses `select()` to create a DataFrame with specific columns: customer_id, amount, and region.

## Aggregations

This cell performs several aggregation operations. It calculates the average transaction amount per region, the total count of fraudulent transactions per region, and the total count for each transaction type.

In [ ]:
df.groupBy("region").avg("amount").show()
df.groupBy("region").sum("is_fraud").show()
df.groupBy("transaction_type").count().show()

+------+------------------+
|region|       avg(amount)|
+------+------------------+
| South|24863.821007751936|
|  East|23983.666198347106|
|  West| 25563.42611111112|
| North|23597.939032258077|
+------+------------------+

+------+-------------+
|region|sum(is_fraud)|
+------+-------------+
| South|           57|
|  East|           59|
|  West|           58|
| North|           67|
+------+-------------+

+----------------+-----+
|transaction_type|count|
+----------------+-----+
|        Purchase|  162|
|          Refund|  172|
|      Withdrawal|  166|
+----------------+-----+



This cell performs several aggregation operations. It calculates the average transaction amount per region, the total count of fraudulent transactions per region, and the total count for each transaction type.

## Spark SQL

This section leverages Spark SQL for querying the DataFrame. First, `df.createOrReplaceTempView("transactions")` registers the DataFrame as a temporary SQL table. Then, SQL queries are executed to count transactions per region, find the top 5 transactions by amount, and sum fraudulent transactions per region.

In [ ]:
df.createOrReplaceTempView("transactions")

spark.sql("SELECT region, COUNT(*) as total FROM transactions GROUP BY region").show()

spark.sql("SELECT * FROM transactions ORDER BY amount DESC LIMIT 5").show()

spark.sql("SELECT region, SUM(is_fraud) as fraud_count FROM transactions GROUP BY region").show()

+------+-----+
|region|total|
+------+-----+
| South|  129|
|  East|  121|
|  West|  126|
| North|  124|
+------+-----+

+--------------+-----------+-----------------+------+--------+----------------+--------+
|transaction_id|customer_id|             name|region|  amount|transaction_type|is_fraud|
+--------------+-----------+-----------------+------+--------+----------------+--------+
|           427|       1036|      Todd Thomas|  West|49969.04|      Withdrawal|       1|
|           235|       1025|       Mark Smith| South|49825.23|          Refund|       0|
|           311|       1081|          Alan Yu|  West|49818.54|        Purchase|       0|
|           117|       1056|     Thomas Smith| South|49629.78|          Refund|       1|
|           305|       1046|Elizabeth Krueger| North|49602.85|        Purchase|       0|
+--------------+-----------+-----------------+------+--------+----------------+--------+

+------+-----------+
|region|fraud_count|
+------+-----------+
| South|      

This section leverages Spark SQL for querying the DataFrame. First, `df.createOrReplaceTempView("transactions")` registers the DataFrame as a temporary SQL table. Then, SQL queries are executed to count transactions per region, find the top 5 transactions by amount, and sum fraudulent transactions per region.

## Optimization

This cell demonstrates optimization techniques. `df.cache()` caches the DataFrame in memory for faster access in subsequent operations. `df.explain()` shows the physical plan of the DataFrame, which can be useful for understanding and optimizing query execution.

In [ ]:
df.cache()
df.explain()

== Physical Plan ==
InMemoryTableScan [transaction_id#17, customer_id#18, name#19, region#20, amount#21, transaction_type#22, is_fraud#23]
   +- InMemoryRelation [transaction_id#17, customer_id#18, name#19, region#20, amount#21, transaction_type#22, is_fraud#23], StorageLevel(disk, memory, deserialized, 1 replicas)
         +- *(1) Filter atleastnnonnulls(7, transaction_id#17, customer_id#18, name#19, region#20, amount#21, transaction_type#22, is_fraud#23)
            +- FileScan csv [transaction_id#17,customer_id#18,name#19,region#20,amount#21,transaction_type#22,is_fraud#23] Batched: false, DataFilters: [atleastnnonnulls(7, transaction_id#17, customer_id#18, name#19, region#20, amount#21, transactio..., Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/spark_lab_dataset.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transaction_id:int,customer_id:int,name:string,region:string,amount:double,transaction_typ...




This cell demonstrates optimization techniques. `df.cache()` caches the DataFrame in memory for faster access in subsequent operations. `df.explain()` shows the physical plan of the DataFrame, which can be useful for understanding and optimizing query execution.